# RAGmap Tutorial // #5dgai Edition 🔵🔴🟡🟢

In this demo, you will learn how to use `ragmap` to explore document chunks and queries in embedding space.

You can use `ragmap` to load, index, store and query documents, just like any RAG workflow.

> ❗ **Didn't get that last sentence?** We strongly advise that you check the [High-Level Concepts](https://docs.llamaindex.ai/en/stable/getting_started/concepts.html) from LlamaIndex's documentation before proceeding.

<img src="https://docs.llamaindex.ai/en/stable/_static/getting_started/basic_rag.png" width="50%"/>

The real power of `ragmap` is that it allows you to *visualize* the embedding space of the documents you load by using dimensionality reduction techniques like [UMAP](https://pair-code.github.io/understanding-umap/) and [t-SNE](https://www.datacamp.com/tutorial/introduction-t-sne).

`ragmap` is built on top of [ChromaDB](https://www.trychroma.com/), which allows us to use either text or embeddings as search queries against our documents.

So here's what we're going to do:
1. Load a document
2. Extract text from the document
3. Split the text into chunks
4. Turn those chunks into embeddings
5. Store the embeddings in a vector database (Chroma)
6. Transform those embeddings into 2D/3D projections
7. Run queries using natural language
8. Plot the results

🤩 **Ready to start?** Let's dig in...

## Step 0: Initial Setup

Let's start by installing the `ragmap` package

In [1]:
!pip install -U git+https://github.com/JGalego/RAGmap.git@feat/pymodule

  Cloning https://github.com/JGalego/RAGmap.git (to revision feat/pymodule) to /private/var/folders/6s/ygl3kzvn6tvdjtvlwm0tgl_00000gn/T/pip-req-build-_41h8eny
  Running command git clone --filter=blob:none --quiet https://github.com/JGalego/RAGmap.git /private/var/folders/6s/ygl3kzvn6tvdjtvlwm0tgl_00000gn/T/pip-req-build-_41h8eny
  Running command git checkout -b feat/pymodule --track origin/feat/pymodule
  Switched to a new branch 'feat/pymodule'
  branch 'feat/pymodule' set up to track 'origin/feat/pymodule'.
  Resolved https://github.com/JGalego/RAGmap.git to commit 8bf503e3c29e0aa1997a908cc3230a0e0a6a7078
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


setting up logging

In [2]:
import sys
import logging

logging.basicConfig(
	stream=sys.stdout,
	level=logging.INFO,
	format='%(levelname)s: %(message)s'
)

and disabling all warnings

In [3]:
import warnings

warnings.simplefilter(action='ignore')

so we can get some visibility into what's happening inside the package.

## Step 1: Initialize

The first thing we need is a **text splitter**, which is responsible for breaking the document text into small, semantically meaningful chunks. 

For now, we will use a [RecursiveCharacterTextSplitter](https://python.langchain.com/docs/modules/data_connection/document_transformers/recursive_text_splitter) from LangChain, which is the recommended one for for generic text.

> If you're looking for hands-on introduction to text splitters, be sure to check the [5 Levels Of Text Splitting](https://github.com/FullStackRetrieval-com/RetrievalTutorials/blob/main/5_Levels_Of_Text_Splitting.ipynb) notebook.

In [4]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

character_splitter = RecursiveCharacterTextSplitter(
	separators=["\n\n", "\n", ". ", " ", ""],
	chunk_size=256,
	chunk_overlap=20
)

Next, we need to create a vector store

In [5]:
import os

from getpass import getpass

os.environ['GOOGLE_API_KEY'] = getpass()

In [6]:
# Opt out on anonymized telemetry
# https://docs.trychroma.com/docs/overview/telemetry
%env ANONYMIZED_TELEMETRY=False

env: ANONYMIZED_TELEMETRY=False


In [7]:
from ragmap.constants import ModelProvider
from ragmap.databases import ChromaDb

vectordb = ChromaDb(
    provider=ModelProvider.GOOGLE_GENAI,
    model="models/text-embedding-004",
    embed_func_kwargs={
        'api_key_env_var': "GOOGLE_API_KEY"
    }
)

INFO: Setting embedding function (model: models/text-embedding-004, provider: Google GenAI 🧠)
INFO: Initializing Chroma collection (name: c0e9815ee85647839a769bd0e5639561, metadata: None)


We can now initialize the RAGmap object by passing the text splitter and vector database.

In [8]:
import ragmap

rm = ragmap.RAGmap(
    text_splitter=character_splitter,
	vectordb=vectordb
)

INFO: Initializing RAGmap


## Step 2: Load + Index

Now, the real magic begins when we start uploading document to the vector data store. 

We'll use a modern rendition of the classic tale [Little Red Riding Hood](https://medium.com/@austenallred/every-amazon-shareholder-letter-as-downloadable-pdf-4eb2ae886018).

<img src="../images/little_red_riding_hood.png" width="50%"/>

In [9]:
rm.load_file('little_red_riding_hood.pdf')

INFO: Loading >>> little_red_riding_hood.pdf
INFO: Extracting text from file
INFO: Retrieving document metadata
INFO: Splitting text into chunks
INFO: Storing data in the vector database


## Step 3: Query

Once the document is indexed, you can start running queries.

In [10]:
vectordb.query(
	query_texts="What did Little Red Riding Hood's mother asked her to do?",
	n_results=3
)['documents'][0][0]

'"Don\'t worry, mommy," said Little Re d Riding Hood, "I\'ll be careful."  \n \nBut when Little Red Riding Hood noticed some lovely flowers in the woods, she forgot her promise to her mother.   She'

## Step 4: Plot

You can use the `plot` method to display the reduced embedding space.

In [11]:
rm.plot()

INFO: Retrieving data from vector store: ['documents']
INFO: Retrieving data from vector store: ['embeddings']
INFO: Generating 2D UMAP projections
INFO: Retrieving vectors from datastore
INFO: Retrieving data from vector store: ['embeddings']
INFO: Fitting 2D UMAP transform to embeddings
INFO: Preparing embeddings projection dataframe for visualization


In [12]:
rm.plot(ragmap.DimensionReduction.TSNE, n_components=3)

INFO: Retrieving data from vector store: ['documents']
INFO: Retrieving data from vector store: ['embeddings']
INFO: Generating 3D TSNE projections
INFO: Retrieving vectors from datastore
INFO: Retrieving data from vector store: ['embeddings']
INFO: Fitting 3D TSNE transform to embeddings
INFO: Precomputed initialization provided. Ignoring initalization-related parameters.
INFO: Automatically determined negative gradient method `bh`
INFO: Automatically determined negative gradient method `bh`
INFO: Automatically determined negative gradient method `bh`
INFO: Automatically determined negative gradient method `bh`
INFO: Preparing embeddings projection dataframe for visualization


In [13]:
rm.plot(
    query=dict(
        query_texts="In the end, what happened to the big bad wolf?",
        n_results=5
    ),
    dimension_reduction_kwargs={
        "random_state": 0,
        "transform_seed": 0
    }
).update_layout(
    width=800,
    height=800
)

INFO: Retrieving data from vector store: ['documents']
INFO: Retrieving data from vector store: ['embeddings']
INFO: Generating 2D UMAP projections
INFO: Retrieving vectors from datastore
INFO: Retrieving data from vector store: ['embeddings']
INFO: Fitting 2D UMAP transform to embeddings
INFO: Running query
{'query_texts': 'In the end, what happened to the big bad wolf?', 'n_results': 5}
INFO: Preparing embeddings projection dataframe for visualization


## Step 999: What's Next?

If you wandered this far, here are some questions to ponder about:

- Did you get the right answer?
- How would you improve the results?
- Why do you think the retrieved chunks are not the ones closest to our query?
- Is there a better way to chunk our document?
- How would you expand the original query?

<img src="https://www.icegif.com/wp-content/uploads/2023/06/icegif-855.gif" width="20%">